## Linux capabilities

Linux splits the all-powerful root into ~40 **capabilities** — individual privileges like "bind a low port" or "change file ownership." Docker's default already drops about half, but the ~14 it keeps still include some that have enabled escapes (`NET_RAW`, `SETUID`, `DAC_OVERRIDE`). The hardened move is to **drop everything, then add back only what the app needs**:

```bash
docker run --cap-drop=ALL --cap-add=NET_BIND_SERVICE myorg/api
```

For most application code the "add back" list is **nothing** — plain code needs no capabilities at all. The handful you might restore:

| Capability | For | Seen in |
|---|---|---|
| `NET_BIND_SERVICE` | binding ports < 1024 | Nginx/Apache as non-root |
| `CHOWN` | `chown` beyond your UID | init scripts fixing ownership |
| `SETUID` / `SETGID` | dropping privileges after start | servers that bind then de-escalate |
| `NET_ADMIN` | configuring interfaces | VPN / network tools |

In Compose it's the same idea, declaratively:

```yaml
services:
  api:
    cap_drop: [ALL]
    cap_add: [NET_BIND_SERVICE]
```

The principle is **least privilege made concrete**: instead of reasoning about which of 40 powers are dangerous, you start from *zero* powers and justify each one you add. It pairs naturally with running non-root — a non-root process with all capabilities dropped can do almost nothing beyond its own job, which is exactly the point. `--cap-drop=ALL` with an empty add list should be your default, relaxed only when the app genuinely fails without a specific capability.